# LMMS using Statsmodels to compare output with Pymer (Lme4)

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels as sm

In [3]:
data_df = pd.read_excel('/Users/thomasgooding/Desktop/Rutgers_SQLdata/physio_demo_data_cleaned_04302026.xlsx')

data_df.head()

,ID6,from_study,timepoint,RRIMin,RRImax,RRImean,RRIRmssd,RRIPnn50,RRIDev,RRICvs,...,RRISUlf_ln,RRISVlf_ln,RRISLf_ln,RRISHf_ln,RRISLfHF_ln,RRISnLf_ln,RRISnHf_ln,RRI_ln,RRIRmssd_ln,RRIPnn50_ln
0,152000,VT1,B1,689,1066,838,39,19.7,53,4.7,...,5.22,5.87,6.35,6.09,4.86,-0.84,NaN,7.29,3.66,2.98
1,152000,VT1,6P,662,1207,882,93,40.5,133,10.5,...,3.18,5.65,9.52,7.88,6.24,-1.82,NaN,14.16,4.53,3.70
2,152002,VT1,B1,673,992,850,33,10.6,57,3.9,...,5.53,6.48,6.56,5.44,5.73,-1.41,NaN,10.69,3.50,2.36
3,152002,VT1,6P,617,1118,878,87,45.5,148,9.9,...,4.61,6.68,9.78,7.49,6.89,-2.39,NaN,14.40,4.47,3.82
4,152004,VT1,B1,702,961,814,24,1.4,38,2.9,...,2.20,5.45,5.91,5.27,5.25,-1.08,NaN,8.36,3.18,0.34


In [4]:
# Explicitly set as ordered categorical with reference levels
data_df['timepoint'] = pd.Categorical(
    data_df['timepoint'], 
    categories=['B1', '6P'],  # B1 first = reference
    ordered=True
)
data_df['bmi_cat'] = pd.Categorical(
    data_df['bmi_cat'],
    categories=['Healthy weight', 'Underweight', 'Overweight', 
                'Obesity_1', 'Obesity_2', 'Obesity_3'],
    ordered=True
)


In [5]:
lmm_df = data_df.copy()

lmm_df.head()

,ID6,from_study,timepoint,RRIMin,RRImax,RRImean,RRIRmssd,RRIPnn50,RRIDev,RRICvs,...,RRISUlf_ln,RRISVlf_ln,RRISLf_ln,RRISHf_ln,RRISLfHF_ln,RRISnLf_ln,RRISnHf_ln,RRI_ln,RRIRmssd_ln,RRIPnn50_ln
0,152000,VT1,B1,689,1066,838,39,19.7,53,4.7,...,5.22,5.87,6.35,6.09,4.86,-0.84,NaN,7.29,3.66,2.98
1,152000,VT1,6P,662,1207,882,93,40.5,133,10.5,...,3.18,5.65,9.52,7.88,6.24,-1.82,NaN,14.16,4.53,3.70
2,152002,VT1,B1,673,992,850,33,10.6,57,3.9,...,5.53,6.48,6.56,5.44,5.73,-1.41,NaN,10.69,3.50,2.36
3,152002,VT1,6P,617,1118,878,87,45.5,148,9.9,...,4.61,6.68,9.78,7.49,6.89,-2.39,NaN,14.40,4.47,3.82
4,152004,VT1,B1,702,961,814,24,1.4,38,2.9,...,2.20,5.45,5.91,5.27,5.25,-1.08,NaN,8.36,3.18,0.34


In [10]:
import statsmodels.formula.api as smf

## make new df — same filtering as pymer version
lmm_df_sm = data_df.dropna(subset=['bmi_cat']).copy()

## Set reference levels explicitly via categorical dtype
lmm_df_sm['timepoint'] = pd.Categorical(
    lmm_df_sm['timepoint'],
    categories=['B1', '6P'],   # B1 = reference
    ordered=False
)
lmm_df_sm['bmi_cat'] = pd.Categorical(
    lmm_df_sm['bmi_cat'],
    categories=['Healthy weight', 'Underweight', 'Overweight',
                'Obesity_1', 'Obesity_2', 'Obesity_3'],  # Healthy weight = reference
    ordered=False
)

## Fit the LMM with random intercept per subject
model_sm = smf.mixedlm(
    formula='HRMean ~ timepoint * bmi_cat',   # fixed effects + interaction
    data=lmm_df_sm,
    groups=lmm_df_sm['ID6']                   # random intercept per subject
)

result_sm = model_sm.fit(reml=True)

print("Statsmodels LMM with B1 reference point")
print(" ")           # REML standard for LMM
print(result_sm.summary())

Statsmodels LMM with B1 reference point
 
                       Mixed Linear Model Regression Results
Model:                     MixedLM          Dependent Variable:          HRMean    
No. Observations:          850              Method:                      REML      
No. Groups:                337              Scale:                       16.2868   
Min. group size:           2                Log-Likelihood:              -2842.3277
Max. group size:           4                Converged:                   Yes       
Mean group size:           2.5                                                     
-----------------------------------------------------------------------------------
                                        Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
-----------------------------------------------------------------------------------
Intercept                               70.933    0.794 89.309 0.000  69.377 72.490
timepoint[T.6P]                          4.365    0.379 1

In [11]:
## make new df — same filtering as pymer version
lmm_df_sm = data_df.dropna(subset=['bmi_cat']).copy()

## Set reference levels explicitly via categorical dtype
lmm_df_sm['timepoint'] = pd.Categorical(
    lmm_df_sm['timepoint'],
    categories=['6P', 'B1'],   # 6P = reference
    ordered=False
)
lmm_df_sm['bmi_cat'] = pd.Categorical(
    lmm_df_sm['bmi_cat'],
    categories=['Healthy weight', 'Underweight', 'Overweight',
                'Obesity_1', 'Obesity_2', 'Obesity_3'],  # Healthy weight = reference
    ordered=False
)

## Fit the LMM with random intercept per subject
model_sm = smf.mixedlm(
    formula='HRMean ~ timepoint * bmi_cat',   # fixed effects + interaction
    data=lmm_df_sm,
    groups=lmm_df_sm['ID6']                   # random intercept per subject
)

result_sm = model_sm.fit(reml=True)           # REML standard for LMM
print("Statsmodels LMM with 6P reference point")
print(" ")         
print(result_sm.summary())

Statsmodels LMM with 6P reference point
 
                       Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          HRMean    
No. Observations:           850              Method:                      REML      
No. Groups:                 337              Scale:                       16.2868   
Min. group size:            2                Log-Likelihood:              -2842.3277
Max. group size:            4                Converged:                   Yes       
Mean group size:            2.5                                                     
------------------------------------------------------------------------------------
                                        Coef.  Std.Err.    z    P>|z|  [0.025 0.975]
------------------------------------------------------------------------------------
Intercept                               75.298    0.794  94.804 0.000  73.741 76.855
timepoint[T.B1]                         -4.365 